# Mini-ETL Package Demo

Welcome to the mini-etl demonstration notebook! This notebook will walk you through the core functions and capabilities of the mini-etl package, a simple but powerful tool for Extract-Transform-Load (ETL) operations.

## What is mini-etl?

The mini-etl package provides a lightweight toolkit for processing financial and time-series data:
- **Extract**: Load data from CSV files
- **Transform**: Clean currency amounts, parse timestamps, and aggregate data
- **Load/Summarize**: Generate daily summary reports

This notebook demonstrates how to use each function individually and how they work together in a complete ETL pipeline.

## Setup and Imports

Let's start by importing the necessary libraries and the mini-etl module.

In [ ]:
import pandas as pd
import math
from pathlib import Path

# Import the mini_etl module
import mini_etl

print("✓ All imports successful!")

## Function 1: Cleaning Currency Amounts

The `clean_amount_series()` function handles messy financial data by:
- Stripping whitespace
- Removing dollar signs ($) and commas (,)
- Converting to numeric values
- Handling invalid data gracefully (converts to NaN)

This is essential for the **Extract** phase when working with formatted financial data.

In [ ]:
# Example: Cleaning messy currency data
messy_amounts = pd.Series([" $1,200.50 ", "$9.99", "  500  ", "not-a-number", "-$15.00"])
print("Original (messy) data:")
print(messy_amounts)
print()

# Apply the cleaning function
clean_amounts = mini_etl.clean_amount_series(messy_amounts)
print("Cleaned data:")
print(clean_amounts)
print()

# Verify types
print(f"Data type: {clean_amounts.dtype}")
print(f"Missing values: {clean_amounts.isna().sum()}")

## Function 2: Parsing Timestamps

The `parse_timestamp_series()` function converts timestamp strings into proper pandas datetime objects:
- Parses ISO-formatted datetime strings
- Handles invalid timestamps gracefully (converts to NaT - "Not a Time")
- Enables time-series operations and aggregations

This function is crucial for the **Transform** phase when working with temporal data.

In [ ]:
# Example: Parsing timestamp data
timestamps = pd.Series([
    "2024-01-01 10:30:00",
    "2024-01-02 14:15:30",
    "2024-01-03 09:00:00",
    "not-a-valid-date",  # This will become NaT
    "2024-01-05 23:59:59"
])
print("Original (string) timestamps:")
print(timestamps)
print(f"Data type: {timestamps.dtype}\n")

# Parse the timestamps
parsed = mini_etl.parse_timestamp_series(timestamps)
print("Parsed timestamps:")
print(parsed)
print(f"Data type: {parsed.dtype}")
print(f"Missing values (NaT): {parsed.isna().sum()}")

## Function 3: Summarizing Data Daily

The `summarize_daily()` function aggregates transaction data into daily summaries:
- Groups records by date (date only, no time component)
- Calculates total amounts per day (NaN values are ignored in sums)
- Counts the number of rows per day (including rows with missing amounts)
- Returns a sorted summary

This is the **Load** phase where raw data is transformed into actionable insights.

In [ ]:
# Example: Aggregating data by day
sample_data = pd.DataFrame({
    "timestamp": pd.to_datetime([
        "2024-01-01 09:00:00",
        "2024-01-01 10:30:00",
        "2024-01-01 15:00:00",  # This row has NaN amount
        "2024-01-02 08:00:00",
        "2024-01-02 12:00:00"
    ]),
    "amount": [100.0, 250.50, float("nan"), 50.0, 75.25]
})

print("Raw data:")
print(sample_data)
print()

# Summarize by day
summary = mini_etl.summarize_daily(sample_data)
print("Daily summary:")
print(summary)
print()
print("Note: 2024-01-01 has 3 rows but total only counts 2 amounts (NaN ignored)")
print("However, num_rows=3 because we count all rows including the one with NaN")

## Full ETL Pipeline Demo

Now let's see how all three functions work together in the complete `run_pipeline()` function. This function:

1. **Extract**: Reads a CSV file with timestamp and amount columns
2. **Transform**: 
   - Cleans the amount column (remove $, commas, handle invalid values)
   - Parses the timestamp column (convert to datetime, handle invalid dates)
   - Drops rows with missing timestamps
3. **Load**: Summarizes the data by day

This is a complete ETL workflow that takes raw, messy data and produces clean, aggregated insights.

In [ ]:
# First, let's examine the sample CSV file
sample_file = Path("data_sample.csv")
print("Sample CSV contents:")
print(sample_file.read_text())
print("\n" + "="*60 + "\n")

In [ ]:
# Run the full pipeline
result = mini_etl.run_pipeline(sample_file)

print("✓ Pipeline completed successfully!\n")
print("Final daily summary:")
print(result)
print()
print("Summary Statistics:")
print(f"  - Date range: {result['date'].min()} to {result['date'].max()}")
print(f"  - Total transactions: {result['num_rows'].sum()}")
print(f"  - Total amount: ${result['total_amount'].sum():.2f}")

## Key Takeaways

The mini-etl package provides a simple, composable set of functions for cleaning and aggregating financial transaction data:

1. **`clean_amount_series()`** - Handles messy currency data by removing formatting characters and validating numeric values
2. **`parse_timestamp_series()`** - Converts string timestamps to proper datetime objects, handling invalid inputs gracefully
3. **`summarize_daily()`** - Aggregates transactions by day, providing total amounts and row counts
4. **`run_pipeline()`** - Orchestrates all three functions into a complete ETL workflow

The pipeline is robust and handles edge cases like:
- Missing or invalid numeric values (becomes NaN)
- Malformed timestamps (becomes NaT)
- Mixed formatting in currency data

This makes it ideal for processing real-world financial data from various sources!